# Rotten Fruit Detector — Colab Training (SSD EfficientDet D1)

Trains **SSD EfficientDet D1** with the TF Object Detection API (`src/dataset.py` + `src/model.py` + `src/train.py`). `main()` builds the label map + TFRecords from the raw Roboflow CSV export, configures the pipeline, then trains, evaluates and exports.

Expected `dataset.zip` (in Drive) contents:

```text
train/_annotations.csv + *.jpg
valid/_annotations.csv + *.jpg
test/_annotations.csv  + *.jpg
```
The split folders may sit at the zip root or one level deep — the notebook auto-detects either.

**Cost tips:** smoke-test with `num_steps=200` first, then the real run. Setup (API install + weights) takes ~5 min of GPU time. Switch variants with `model_name="efficientdet_d0"` (cheaper) or `"efficientdet_d5"` (pricier). Colab gives one GPU (usually a T4), so keep `batch_size` small (4). Set **Runtime → Change runtime type → GPU** first.

In [ ]:
# 1. Clone the repo and install system + python deps
REPO_URL = "https://github.com/YOUR_USERNAME/rotten-fruit-detection-ai.git"

!git clone {REPO_URL} /content/rotten-fruit-detection-ai
%cd /content/rotten-fruit-detection-ai
!pip install -q -r requirements.txt

In [ ]:
# 2. Mount Drive, unzip the raw dataset, and locate the split folders
from google.colab import drive
import zipfile
from pathlib import Path

drive.mount("/content/drive")

EXTRACT_DIR = Path("dataset")
with zipfile.ZipFile("/content/drive/MyDrive/dataset.zip") as archive:
    archive.extractall(EXTRACT_DIR)

# The zip may hold train/valid/test at its root OR nested one folder deep,
# so find the directory that actually contains the split folders.
def find_dataset_root(base):
    base = Path(base)
    for path in [base, *(p for p in base.rglob("*") if p.is_dir())]:
        if (path / "train").is_dir():
            return path
    raise FileNotFoundError(f"No 'train' folder found under {base}")

DATA_DIR = find_dataset_root(EXTRACT_DIR)
print("Dataset root:", DATA_DIR)
print("Contains:", sorted(p.name for p in DATA_DIR.iterdir() if p.is_dir()))

In [ ]:
# 3. Prepare TFRecords, configure EfficientDet D1, then train -> evaluate -> export
#    First run also clones the Object Detection API and downloads the weights.
#    Smoke test first: num_steps=200. Lower batch_size to 2 if you hit OOM.
import os
import sys
sys.path.insert(0, "src")

os.environ["TF_USE_LEGACY_KERAS"] = "1"  # TFOD API needs Keras 2 (tf-keras)

from train import main

main(data_dir=DATA_DIR, model_name="efficientdet_d1", num_steps=4000, batch_size=4)

In [ ]:
# 4. Copy the exported model back to Drive
!cp -r exported-models/efficientdet_d1 /content/drive/MyDrive/